In [ ]:
 #======================================================================
# BLOQUE FINAL: entrenar modelo HGB Poisson para la API y guardar artefactos
# ======================================================================

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_poisson_deviance
from sklearn.linear_model import ElasticNet, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Cargar y preparar los datos exactamente como en main()
df = pd.read_parquet(PARQUET_IN)
df = _build_fecha(df)
df = _normalizar(df)
df = _filtrar_naturales(df)

# 2. Construir el panel
panel = _armar_panel(df)
print(panel.head(3).to_string(index=False))
print(f"\nPanel listo para API: {panel.shape[0]:,} filas × {panel.shape[1]} columnas")

# 3. Definir variable objetivo y matriz de features
y = panel["y_conteo"]
X = panel.copy()

no_features = [DEP_COL, "FECHA", "y_conteo"]
feat_cols = [c for c in X.columns if c not in no_features]
print(f"Número de columnas de features para la API: {len(feat_cols)}")

# Limpiar infinities / NaN igual que en evaluar_modelo
X_feats = X[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Entrenar modelo HGB Poisson sobre TODO el panel 
hgb_api = HistGradientBoostingRegressor(
    loss="poisson",
    learning_rate=0.08,
    max_depth=6,
    max_iter=400,
    l2_regularization=0.0,
)

print("\nEntrenando modelo HGB Poisson final para la API...")
hgb_api.fit(X_feats, y)
print("✅ Modelo HGB Poisson entrenado sobre todo el panel.")

# 5. Construir panel compacto para la API con ANO y MES
panel_api = panel.copy()
panel_api["FECHA"] = pd.to_datetime(panel_api["FECHA"])
panel_api["ANO"] = panel_api["FECHA"].dt.year.astype(int)
panel_api["MES"] = panel_api["FECHA"].dt.month.astype(int)

cols_to_save = [DEP_COL, "ANO", "MES"] + feat_cols
panel_api = panel_api[cols_to_save].drop_duplicates()
print(f"Panel API compacto: {panel_api.shape[0]:,} filas × {panel_api.shape[1]} columnas")

# 6. Guardar artefactos para la API
joblib.dump(hgb_api, "hgb_poisson_model.pkl")
joblib.dump(feat_cols, "hgb_feat_cols.pkl")
panel_api.to_parquet("panel_features.parquet", index=False)

print("\n✅ Artefactos guardados para la API:")
print("   • hgb_poisson_model.pkl")
print("   • hgb_feat_cols.pkl")
print("   • panel_features.parquet")
#